# 08 - Laboratorio de Desarrollo: De Notebook a Hugging Face Space

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Asignatura Ministerial:** Procesamiento Digital de Imágenes
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

El objetivo de este laboratorio es guiar la transición de un flujo de trabajo experimental en Jupyter Notebook hacia una aplicación web interactiva y modular en producción, utilizando el ecosistema de Hugging Face Spaces y Gradio.

## Resultados de aprendizaje

Al final de este notebook ustedes van a poder:
1. Comprender la arquitectura de Hugging Face Hub y la integración con Spaces.
2. Convertir código de Jupyter Notebook a scripts Python modulares y portables.
3. Diseñar interfaces web interactivas utilizando Gradio.
4. Configurar metadatos, requerimientos y dependencias en entornos de producción.
5. Conectar repositorios locales a Hugging Face Spaces mediante Git para despliegue automático.

## Terminología clave (Microglosario)

*   **🤗 Hugging Face Spaces:** Servicio de hosting gratuito para aplicaciones interactivas de machine learning que automatiza la construcción y el despliegue del software.
*   **Gradio:** Framework de desarrollo ágil en Python que permite construir interfaces gráficas web para modelos sin requerir conocimientos de frontend (HTML/CSS/JS).
*   **app.py:** Punto de entrada obligatorio y estándar que Hugging Face busca para ejecutar la aplicación web.
*   **requirements.txt:** Archivo de configuración que declara las dependencias y librerías necesarias con sus versiones para recrear el entorno de ejecución.
*   **Metadata YAML:** Bloque de configuración inicial en el README.md que le indica a Hugging Face qué SDK utilizar y la configuración del Space.

## 1. 🤗 Hugging Face en Profundidad

### ¿Qué es Hugging Face?

**Hugging Face** representa una infraestructura central en la IA moderna (el "GitHub de la IA"), democratizando el acceso a herramientas clave:

*   **Models:** Cientos de miles de modelos preentrenados listos para su uso.
*   **Datasets:** Repositorios de datos públicos y estructurados.
*   **Spaces:** Hosting gratuito de aplicaciones interactivas de Machine Learning.

### Arquitectura Conceptual de Hugging Face

```
┌─────────────────────────────────────────────────────────┐
│                    HUGGING FACE HUB                     │
├─────────────────────────────────────────────────────────┤
│                                                         │
│  ┌─────────────┐  ┌──────────────┐  ┌──────────────┐  │
│  │   MODELS    │  │   DATASETS   │  │    SPACES    │  │
│  │             │  │              │  │              │  │
│  │ • ViT       │  │ • ImageNet   │  │ • Gradio     │  │
│  │ • CLIP      │  │ • COCO       │  │ • Streamlit  │  │
│  │ • DETR      │  │ • Custom     │  │ • Static     │  │
│  └─────────────┘  └──────────────┘  └──────────────┘  │
│                                                         │
│  ┌──────────────────────────────────────────────────┐  │
│  │         INTEGRACIÓN CON GITHUB                   │  │
│  │  • Sincronización automática                     │  │
│  │  • CI/CD integrado                               │  │
│  │  • Versionado de modelos                         │  │
│  └──────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────┘
```

### ¿Qué es un Space?

Un **Space** es una instancia de servidor web configurada y desplegada automáticamente a partir de sus archivos de código fuente. Admite diferentes tecnologías:

*   **Gradio:** Interfaces robustas e interactivas de Python (nuestro foco hoy).
*   **Streamlit:** Cuadros de mando y dashboards analíticos.
*   **Docker:** Contenedores aislados y flexibles para configuraciones a medida.

## 2. Anatomía de un Hugging Face Space

### Estructura de Archivos Obligatoria

Para que la plataforma compile su aplicación correctamente, deben respetar la siguiente estructura mínima en la raíz del repositorio:

```
mi-space/
├── app.py              # ARCHIVO PRINCIPAL - Punto de entrada de la aplicación
├── requirements.txt    # DEPENDENCIAS - Librerías de Python necesarias
├── README.md          # METADATOS Y DOCUMENTACIÓN - Configuración del Space (YAML)
└── .gitignore         # Archivos que deben excluirse del control de versiones
```

### El Archivo app.py: Corazón del Space

El script principal debe denominarse estrictamente `app.py`. A continuación, analizamos una plantilla base didáctica que implementa un clasificador general de imágenes usando la arquitectura **Vision Transformer (ViT)**.

In [ ]:
# Ejemplo conceptual de app.py

import gradio as gr
from transformers import pipeline
from PIL import Image

print('✦ Inicializando la aplicación y cargando recursos de Deep Learning...')

# 1. CARGA DEL MODELO: Se ejecuta una única vez al levantar el contenedor para evitar sobrecostos en cada consulta
clasificador = pipeline(
    'image-classification',
    model='google/vit-base-patch16-224'
)
print('✓ Modelo Vision Transformer cargado y listo.')

# 2. FUNCIÓN DE NEGOCIO: Procesa la entrada recibida desde el frontend interactivo
def clasificar_imagen(imagen):
    """
    Procesa la imagen provista por el usuario y estima las categorías.
    """
    if imagen is None:
        return {'Error': 'No se cargó ninguna imagen'}
        
    try:
        # Inferencia con ViT
        resultados = clasificador(imagen)
        
        # Formateamos los resultados en el diccionario que espera gr.Label {etiqueta: score}
        predicciones_formateadas = {}
        for res in resultados[:5]:
            predicciones_formateadas[res['label']] = float(res['score'])
            
        return predicciones_formateadas
    except Exception as e:
        return {'Error en inferencia': str(e)}

# 3. INTERFAZ GRADIO: Definimos el diseño de la UI
demo = gr.Interface(
    fn=clasificar_imagen,
    inputs=gr.Image(type='pil', label='Suban una imagen'),
    outputs=gr.Label(num_top_classes=5, label='Predicciones de ImageNet'),
    title='Clasificador de Imágenes con ViT',
    description='Carguen una imagen para que el modelo estime su categoría conceptual de ImageNet.',
)

# 4. LANZAMIENTO DE LA APLICACIÓN
if __name__ == '__main__':
    print('✦ Iniciando el servidor local de Gradio...')
    demo.launch()

### Los Archivos Complementarios de Configuración

#### requirements.txt
Declara los paquetes exactos de Python para configurar el contenedor:

```txt
transformers>=4.35.0
torch>=2.0.0
pillow>=9.0.0
gradio>=4.0.0
```

#### README.md con Frontmatter YAML
Hugging Face lee la sección superior encerrada entre `---` para parametrizar el entorno de ejecución en la nube. Observen la estructura:

```markdown
---
title: Clasificador de Imagenes ViT
emoji: ✦
colorFrom: slate
colorTo: blue
sdk: gradio
sdk_version: 4.44.1
app_file: app.py
pinned: false
license: mit
---

# Clasificador de Imágenes con Vision Transformer
Esta aplicación web permite clasificar imágenes en tiempo real utilizando Google ViT.
```

## 3. De Jupyter Notebook a Script Python Profesional

### Diferencias Clave de Ejecución y Flujo

| Aspecto | Jupyter Notebook | Script Python (`app.py`) |
| :--- | :--- | :--- |
| **Ejecución** | Celda por celda (estado en memoria) | De arriba hacia abajo (secuencial completo) |
| **Interactividad** | Excelente para experimentar y graficar | Declarativo y preparado para producción |
| **Imports** | Frecuentemente dispersos en celdas | Concentrados obligatoriamente en la cabecera |
| **Salidas** | Impresiones automáticas en celdas | Deben declararse explícitamente (`print`, `gr.Label`) |

### Patrones de Conversión Didáctica

1. **Remoción de Comandos de Sistema:** Los comandos como `!pip install` o `!unzip` no deben ejecutarse en `app.py`. Deben trasladarse a `requirements.txt` o ser gestionados antes de desplegar.
2. **Eliminación de Solicitudes Interactivas por Consola:** El uso de `input()` de consola bloquea los servidores web. Deben sustituirse por componentes de entrada gráfica de Gradio (`gr.Textbox()`, `gr.Image()`).
3. **Refactorización de Visualizaciones:** En lugar de llamar a `plt.show()`, la lógica debe retornar la imagen o los resultados formateados para que la interfaz los renderice de forma nativa.

## 4. Desarrollo con VS Code y KiloCode (VibeCoding)

### VS Code como Entorno Unificado
Para trabajar eficientemente en proyectos de Visión Artificial en producción, la transición del notebook experimental a scripts de Python se realiza con mayor agilidad en un entorno de desarrollo integrado (IDE) como **VS Code**. Este nos brinda control directo de versión de Git, terminal integrada y depuradores de variables en tiempo de ejecución.

### KiloCode y el Concepto de VibeCoding
El **VibeCoding** es un enfoque didáctico e iterativo que propone apoyarse en asistentes inteligentes (como **KiloCode** y su modelo **CodeSupernova**) para agilizar tareas operativas repetitivas y estructurar el diseño de software. Se basa en tres pilares:

*   **Checkpoints Claros:** Establecer metas pequeñas y probarlas antes de continuar.
*   **Definición Explícita de Contexto:** Proveer al asistente la estructura exacta de entradas y salidas de las funciones.
*   **Depuración Sistemática:** Analizar excepciones mediante logs claros y trazar el error hasta la capa correspondiente.

## 5. Arquitectura Didáctica de 3 Capas

Cuando diseñamos interfaces interactivas para modelos de Visión Digital, es fundamental estructurar el código siguiendo la arquitectura de tres capas para garantizar que sea modular y legible:

```
┌─────────────────────────────────────────────────────────────┐
│             PRESENTATION LAYER (Gradio UI)                 │ ← Interfaz gráfica de usuario
├─────────────────────────────────────────────────────────────┤
│             BUSINESS LOGIC (Funciones de negocio)          │ ← Validación, formateo y control
├─────────────────────────────────────────────────────────────┤
│             DATA LAYER (Modelos deep learning / APIs)       │ ← Carga de pesos e inferencia con PyTorch
└─────────────────────────────────────────────────────────────┘
```

A continuación, presentamos un ejemplo completo de código modular aplicando este principio estructural.

In [ ]:
import os
import gradio as gr
from transformers import pipeline
from PIL import Image

print('✦ Configurando las capas de la aplicación...')

# =====================================================================
# 1. DATA LAYER (Capa de Datos): Carga de Modelos e Inferencia
# =====================================================================
print('✦ [Data Layer] Cargando modelo preentrenado...')
clasificador_modelo = pipeline(
    'image-classification',
    model='google/vit-base-patch16-224'
)
print('✓ [Data Layer] Modelo cargado con éxito en memoria.')

# =====================================================================
# 2. BUSINESS LOGIC LAYER (Capa de Lógica de Negocio): Validaciones y Control
# =====================================================================
def procesar_y_clasificar(imagen):
    """
    Lógica de control. Valida los datos y ejecuta el flujo.
    """
    # Validamos entrada nula
    if imagen is None:
        print('✗ [Business Layer] Intento de clasificación sin datos de imagen.')
        return {'Error': 'Por favor, carguen una imagen válida en el visor.'}
        
    print('✦ [Business Layer] Muestra recibida. Ejecutando preprocesamiento e inferencia...')
    
    try:
        # Inferencia directa mediante la Capa de Datos
        resultados = clasificador_modelo(imagen)
        
        # Formateo de salida didáctica
        salida_diccionario = {}
        for res in resultados[:5]:
            salida_diccionario[res['label']] = float(res['score'])
            
        print('✓ [Business Layer] Procesamiento y estimación completados.')
        return salida_diccionario
        
    except Exception as error:
        print(f'✗ [Business Layer] Falla en ejecución del pipeline: {error}')
        return {'Error de Inferencia': str(error)}

# =====================================================================
# 3. PRESENTATION LAYER (Capa de Presentación): Interfaz de Gradio (Blocks)
# =====================================================================
print('✦ [Presentation Layer] Construyendo interfaz web...')

with gr.Blocks(theme=gr.themes.Soft()) as interfaz_app:
    gr.Markdown('# ✦ Clasificador de Imágenes de Elite (ViT)')
    gr.Markdown('Carguen una imagen en la columna izquierda y hagan clic en **Clasificar muestra**.')
    
    with gr.Row():
        # Columna de Entrada
        with gr.Column():
            imagen_input = gr.Image(type='pil', label='Muestra Visual (Entrada)')
            boton_ejecutar = gr.Button('Clasificar muestra', variant='primary')
            
        # Columna de Salida
        with gr.Column():
            etiqueta_output = gr.Label(num_top_classes=5, label='Estimaciones de Confianza (Salida)')
            
    # Vinculamos la acción del botón a la lógica de negocio
    boton_ejecutar.click(
        fn=procesar_y_clasificar,
        inputs=imagen_input,
        outputs=etiqueta_output
    )
    
print('✓ [Presentation Layer] UI inicializada correctamente.')

## 6. Despliegue e Integración Continua (GitHub + Hugging Face)

### Workflow y Sincronización en Producción
El despliegue en Hugging Face Spaces puede automatizarse de forma directa vinculando Git. Esto les permite programar localmente y actualizar su sitio web público con un simple comando de control de versiones.

### Comandos Esenciales de Despliegue

```bash
# 1. Inicializamos el repositorio Git local en su directorio de trabajo
git init

# 2. Vinculamos el Space remoto de Hugging Face como origen secundario
git remote add origin https://huggingface.co/spaces/TU_USUARIO_HF/NOMBRE_DE_SU_SPACE

# 3. Añadimos todos los archivos requeridos
git add app.py requirements.txt README.md

# 4. Creamos el commit confirmando la versión estable
git commit -m "feat: Primera version funcional del clasificador ViT"

# 5. Empujamos al repositorio remoto en la rama principal
git push -u origin main
```

## Consigna de Lectura e Interpretación

### Preguntas para pensar:
1. ¿Por qué es una buena práctica didáctica y computacional separar la carga de pesos del modelo (`pipeline`) de la función de procesamiento interactivo?
2. Si la aplicación en Hugging Face arroja un error del tipo `ModuleNotFoundError`, ¿dónde reside el problema y qué archivo de configuración deben modificar para subsanarlo?
3. Imaginen que la aplicación web tarda más de 20 segundos en responder para cada imagen procesada. Teniendo en cuenta la arquitectura de 3 capas, ¿qué optimizaciones técnicas podrían proponer para mitigar esta latencia?

## Cierre del Laboratorio

En esta sesión consolidamos los conceptos clave para realizar la transición de modelos de deep learning desde un Jupyter Notebook hacia una interfaz web interactiva en producción. Comprendieron los pilares de la arquitectura de Hugging Face Spaces, el diseño modular en 3 capas de Gradio y la potencia del desarrollo asistido iterativo.

¡Los felicitamos por dar el paso hacia la creación de aplicaciones prácticas y de alto valor en Procesamiento Digital de Imágenes!